# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  
** Ciara Graves**

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** item, i.e. line in order.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7

print(lines)

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


**Q1 Explanation:** We use `record_path`to determine what each line is, i.e. one row per item. Then with `meta` we can find each order and vendor corresponds with each item.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True)

# Find unmatched vendor
unmatched = joined[joined['_merge'] == 'left_only']

print("Unmatched vendor:", unmatched['vendor_id'].unique())
print("Units attached:", unmatched['qty'].sum())

Unmatched vendor: ['V-07']
Units attached: 3


**Q2 Explanation:** Using a left join, we merge the vendor information. This gives us that we have an unmatched vendor V-07 with 3 units.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
# choose to label unmatched vendors as "Unkown" so it is names even if we don't know the name
joined['name'] = joined['name'].fillna('Unknown')

qty_by_vendor = (
    joined.groupby('name')['qty']
          .sum()
          .reset_index())

print(qty_by_vendor)

           name  qty
0     Cav Merch    1
1  Hoos Burgers    3
2       Unknown    3


**Q3: Explanation** First I named the unmatched vendor to be "Unknown" so that it could still be excluded in our quantity by vendor calculation. I didn't exclude it because it provides information.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
zone_totals = (
    joined.groupby('zone')['qty']
          .sum()
          .sort_values(ascending=False))
print(zone_totals)
top_zone = zone_totals.idxmax()
zone_units = zone_totals.sum()

print("Zone with most items:", top_zone)
print("Units included in zone totals:", zone_units)
print("Missing units:", 7 - zone_units) # hard code total to be 7

zone
A    4
Name: qty, dtype: int64
Zone with most items: A
Units included in zone totals: 4
Missing units: 3


**Q4 Explanation:** Here we have that zone A sold the most units with 4 total units, rather than 7. This is because the unmatched vendor has 3 units and it has no zone, so therefore the 3 units went unaccounted for. No matter if the vendor is in zone A, zone A still has sold the most units.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
# need to first handle the missingness
# add in empty lines list for orders missing the key
for order in ragged_json:
    if not order.get('lines'):
        order['lines'] = [{'item': None, 'qty': None}]

# flatten the JSON
ragged = pd.json_normalize(
    ragged_json,
    record_path='lines',
    meta=['order', 'vendor_id'],
    errors='ignore')
print(ragged)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1   None  NaN     5      V-10
2  Fries  NaN     6      V-01


**Q5 Explanation:** Here we use `order.setdefault('lines', [])` to loop through and fill in empty lines before using pandas. Orders with no line items won't create any rows, whereas line items missing a quanitity with have NaN; differentiating it from a true 0 value. This is a distinguishable difference between missing data and the quantity zero.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


**Q6 Explanation:** All checks pass, code is validated.

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**Answers:**
(a) The "one row per line item" is a useful shape to work with because it allows for easy calculation of each item. If we had more than one item in a row, finding a total like total sold would be more difficult.

(b) In this lab a missing quantity means the value is unknown, but a quantity of zero means that no units sold. If we treat quantities as zeros then we are deflating the total sales, giving inaccurate results.